# 9.3 TensorFlow.js

本 notebook 示範如何準備 TensorFlow.js 部署素材：訓練 Keras 模型、保存 metadata、檢查 `tensorflowjs` converter、產生轉換指令與前端推論範本。


## 1. 載入套件


In [ ]:
import json
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(tf.__version__)
import importlib.util
import shutil
import subprocess


## 2. 建立資料並訓練模型


In [ ]:
X, y = make_classification(
    n_samples=1500,
    n_features=10,
    n_informative=7,
    n_redundant=2,
    n_classes=3,
    n_clusters_per_class=1,
    class_sep=1.2,
    flip_y=0.03,
    random_state=SEED,
)
X = X.astype('float32')
y = y.astype('int64')
class_names = ['low', 'medium', 'high']

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_valid = scaler.transform(x_valid).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

tf.keras.backend.clear_session()
tf.random.set_seed(SEED)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],), name='features'),
    tf.keras.layers.Dense(48, activation='relu'),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(len(class_names), activation='softmax', name='probabilities'),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.fit(
    x_train,
    y_train,
    validation_data=(x_valid, y_valid),
    epochs=22,
    batch_size=32,
    verbose=0,
)
print(dict(zip(model.metrics_names, model.evaluate(x_test, y_test, verbose=0))))


## 3. 儲存 Keras 模型與 Metadata


In [ ]:
artifact_dir = Path('artifacts/9_3_tfjs')
tfjs_dir = artifact_dir / 'tfjs_model'
artifact_dir.mkdir(parents=True, exist_ok=True)

keras_model_path = artifact_dir / 'classifier.keras'
model.save(keras_model_path)

metadata = {
    'class_names': class_names,
    'feature_count': int(x_train.shape[1]),
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'example_raw_features': X[0].astype(float).tolist(),
}
metadata_path = artifact_dir / 'metadata.json'
metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(keras_model_path)
print(metadata_path)


## 4. 檢查 TensorFlow.js Converter

如果環境已安裝 `tensorflowjs`，就直接轉換；若沒有安裝，仍會輸出可複製的轉換指令。


In [ ]:
converter_module_found = importlib.util.find_spec('tensorflowjs') is not None
converter_bin = shutil.which('tensorflowjs_converter')
print({'tensorflowjs_module': converter_module_found, 'tensorflowjs_converter': converter_bin})

command = [
    'tensorflowjs_converter',
    '--input_format=keras',
    str(keras_model_path),
    str(tfjs_dir),
]
print(' '.join(command))

if converter_bin:
    if tfjs_dir.exists():
        shutil.rmtree(tfjs_dir)
    completed = subprocess.run(command, check=True, capture_output=True, text=True)
    print(completed.stdout)
else:
    tfjs_dir.mkdir(parents=True, exist_ok=True)
    conversion_note = {
        'note': 'Install tensorflowjs and run the converter command printed above to generate model.json and weight shards.',
        'converter_command': ' '.join(command),
    }
    (tfjs_dir / 'CONVERSION_REQUIRED.json').write_text(
        json.dumps(conversion_note, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

sorted(p.name for p in tfjs_dir.iterdir())


## 5. 產生前端推論範本


In [ ]:
metadata_json = json.dumps(metadata, ensure_ascii=False)
html = f"""<!doctype html>
<html lang=\"zh-Hant\">
<head>
  <meta charset=\"utf-8\" />
  <title>TensorFlow.js Inference Demo</title>
  <script src=\"https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@latest/dist/tf.min.js\"></script>
</head>
<body>
  <h1>TensorFlow.js Inference Demo</h1>
  <pre id=\"output\">Loading model...</pre>
  <script>
    const metadata = {metadata_json};

    function standardize(rawFeatures) {{
      return rawFeatures.map((value, index) => (
        (value - metadata.scaler_mean[index]) / metadata.scaler_scale[index]
      ));
    }}

    async function run() {{
      const model = await tf.loadLayersModel('./tfjs_model/model.json');
      const rawFeatures = metadata.example_raw_features;
      const scaled = standardize(rawFeatures);
      const input = tf.tensor2d([scaled]);
      const probabilities = await model.predict(input).data();
      const bestIndex = probabilities.indexOf(Math.max(...probabilities));
      document.getElementById('output').textContent = JSON.stringify({{
        predicted_class: bestIndex,
        predicted_label: metadata.class_names[bestIndex],
        probabilities: Array.from(probabilities),
      }}, null, 2);
    }}

    run().catch(error => {{
      document.getElementById('output').textContent = error.stack;
    }});
  </script>
</body>
</html>
"""

html_path = artifact_dir / 'index.html'
html_path.write_text(html, encoding='utf-8')
print(html_path)


## 6. 檢查匯出素材


In [ ]:
export_files = []
for path in sorted(artifact_dir.rglob('*')):
    if path.is_file():
        export_files.append({
            'path': str(path.relative_to(artifact_dir)),
            'size_kb': round(path.stat().st_size / 1024, 2),
        })
pd.DataFrame(export_files)


## 7. 套用自己的資料

TensorFlow.js 部署時，請把模型檔、metadata、class names 與前處理參數一起交給前端。若目前環境沒有 converter，可先在本機或 CI 安裝 `tensorflowjs` 後執行本 notebook 印出的轉換指令。
